# CircuitSight — Fine-Tuning & Evaluation (Qwen2.5-VL-3B, QLoRA)

Trains a small vision-language model to **read a circuit schematic and solve it** (components → topology → equations → values), using the dataset produced by `CircuitSight_dataset_generation.ipynb`.

**Run order:**
1. Setup + GPU check
2. Load the dataset (from Drive or an uploaded zip)
3. Load Qwen2.5-VL-3B (4-bit) + LoRA
4. **Day-1 OOM smoke test — the go/no-go gate.** Run this *before* committing to a full run. If it OOMs, follow the fallback notes (lower resolution or smaller model) before proceeding.
5. Full training
6. Inference
7. **Evaluation harness** — scores base vs. tuned on: component accuracy, R_eq accuracy, final-answer accuracy, and — separately — **fabrication rate** vs. **honest-abstention rate**.

Needs a GPU runtime (Runtime → Change runtime type → GPU). With Colab credits, an **A100 40GB** is comfortable for the 3B; a T4/L4 works at lower resolution or with SmolVLM.


## 1. Setup

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          "|", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")
else:
    print("No GPU! Runtime -> Change runtime type -> GPU before continuing.")

In [ ]:
# Unsloth install (Colab). If Colab has torch issues on the day, use Unsloth's --no-deps recipe.
!pip -q install unsloth 2>/dev/null
!pip -q install --no-deps --upgrade unsloth unsloth_zoo 2>/dev/null
from unsloth import FastVisionModel, is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
print("unsloth ready")

## 2. Config

In [ ]:
CFG = dict(
    MODEL       = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit",  # fallback: SmolVLM / a 2B if VRAM is tight
    MAX_IMAGE_PX= 768,     # longest image side; the main VRAM knob for a VLM
    LORA_R      = 16, LORA_ALPHA = 16,
    BATCH       = 1, GRAD_ACCUM = 4,
    EPOCHS      = 1,       # 1 epoch over 10k-50k is plenty; watch the eval curve
    LR          = 2e-4,
    MAX_LEN     = 2048,
    SEED        = 3407,
    DATA_DIR    = "circuitsight_dataset",   # unzipped dataset folder
    OUT_DIR     = "circuitsight_qlora",
)
INSTRUCTION = ("You are a circuit analysis tutor. Look at the schematic and answer the question. "
    "First list the components, then state the topology (what is in series/parallel), then write "
    "the equations and solve step by step. If a component value is not legible, say so and report "
    "it as null instead of guessing. End with a line 'FINAL: {json}'.")
CFG

## 3. Load the dataset

Get the dataset next to this notebook. Either mount Drive and point `DATA_DIR` at the unzipped folder, or upload the zip produced by the data-gen notebook.

In [ ]:
import os, json, zipfile
from datasets import Dataset
from PIL import Image

# Option A: upload the zip (uncomment)
# from google.colab import files; up = files.upload()
# zname = next(iter(up)); zipfile.ZipFile(zname).extractall(".")

# Option B: Google Drive (uncomment)
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/drive/MyDrive/circuitsight_dataset.zip . && unzip -q circuitsight_dataset.zip -d circuitsight_dataset

assert os.path.isdir(CFG["DATA_DIR"]), f"{CFG['DATA_DIR']} not found - upload/unzip the dataset first"
IMG_DIR = os.path.join(CFG["DATA_DIR"], "images")

def load_jsonl(name):
    p = os.path.join(CFG["DATA_DIR"], name)
    return [json.loads(l) for l in open(p)] if os.path.exists(p) else []

train_rows = load_jsonl("train.jsonl")
val_rows   = load_jsonl("val_synthetic.jsonl")
print(f"train: {len(train_rows)}  val: {len(val_rows)}")
print("example keys:", list(train_rows[0].keys()))

## 4. Format as vision chat samples

Each row → a user turn (image + instruction + question) and an assistant turn (the worked solution). Images are loaded as RGB PIL and downsized to `MAX_IMAGE_PX` — passing real PIL images (not paths) avoids the common *'could not make a flat list of images'* collator error.

In [ ]:
def load_image(name):
    img = Image.open(os.path.join(IMG_DIR, name)).convert("RGB")
    m = CFG["MAX_IMAGE_PX"]
    if max(img.size) > m:
        s = m / max(img.size); img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
    return img

def to_conversation(row):
    return {"messages": [
        {"role":"user","content":[
            {"type":"image","image": load_image(row["image"])},
            {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + row["question"]}]},
        {"role":"assistant","content":[{"type":"text","text": row["target_output"]}]},
    ]}

train_conv = [to_conversation(r) for r in train_rows]
print("formatted", len(train_conv), "samples")
print("sample user text:\n", train_conv[0]["messages"][0]["content"][1]["text"][:200])

## 5. Load model + attach LoRA

We finetune both vision and language layers: component *identification* is a vision-side skill, equation *setup* is language-side, and this task needs both.

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    CFG["MODEL"], load_in_4bit=True, use_gradient_checkpointing="unsloth")
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=CFG["LORA_R"], lora_alpha=CFG["LORA_ALPHA"], lora_dropout=0,
    bias="none", random_state=CFG["SEED"])
print("model + LoRA ready")

## 6. Day-1 OOM smoke test — the go/no-go gate

**Run this before the full training.** It trains 5 steps on a handful of samples and reports peak VRAM. Purpose: find out *today* whether the model trains without OOM at your image resolution, instead of discovering it 3 days in.

- **Passes** (completes, peak VRAM leaves headroom) → proceed to full training.
- **OOMs** → in order: (1) drop `MAX_IMAGE_PX` to 512 and re-run cell 4; (2) set `finetune_vision_layers=False`; (3) switch `CFG["MODEL"]` to a 2B (e.g. SmolVLM) and reload cell 5. Re-run this gate until it passes.


In [ ]:
import time
FastVisionModel.for_training(model)
smoke = SFTTrainer(
    model=model, tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_conv[:8],
    args=SFTConfig(
        per_device_train_batch_size=1, gradient_accumulation_steps=1,
        warmup_steps=0, max_steps=5, learning_rate=CFG["LR"], logging_steps=1,
        optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear",
        seed=CFG["SEED"], output_dir="smoke_out", report_to="none",
        remove_unused_columns=False, dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True}, max_length=CFG["MAX_LEN"],
    ))
torch.cuda.reset_peak_memory_stats()
t=time.time(); smoke.train(); dt=time.time()-t
peak = torch.cuda.max_memory_reserved()/1e9
total = torch.cuda.get_device_properties(0).total_memory/1e9
print(f"\nSMOKE TEST PASSED: 5 steps in {dt:.0f}s | peak VRAM {peak:.1f} / {total:.1f} GB")
print("GO." if peak < 0.9*total else "TIGHT - lower MAX_IMAGE_PX before the full run.")

## 7. Full training

One epoch is usually enough for this narrow task. **Scale strategy:** start with ~10-15k records, run, look at the eval curve in section 9, and only scale toward 50k if accuracy is still climbing — more data past diminishing returns just burns credits.

In [ ]:
FastVisionModel.for_training(model)
steps_per_epoch = max(1, len(train_conv)//(CFG["BATCH"]*CFG["GRAD_ACCUM"]))
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_conv,
    args=SFTConfig(
        per_device_train_batch_size=CFG["BATCH"], gradient_accumulation_steps=CFG["GRAD_ACCUM"],
        warmup_steps=10, num_train_epochs=CFG["EPOCHS"], learning_rate=CFG["LR"],
        logging_steps=25, optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear",
        seed=CFG["SEED"], output_dir=CFG["OUT_DIR"], report_to="none",
        remove_unused_columns=False, dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True}, max_length=CFG["MAX_LEN"],
        save_steps=500, save_total_limit=2,
    ))
stats = trainer.train()
print("done. final loss:", round(stats.training_loss, 4))

## 8. Inference helper

In [ ]:
def solve_image(pil_img, question, model, tokenizer, max_new_tokens=400):
    FastVisionModel.for_inference(model)
    msgs = [{"role":"user","content":[{"type":"image"},
             {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + question}]}]
    text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = tokenizer(pil_img, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True,
                         do_sample=False, temperature=0.0)
    return tokenizer.decode(out[0], skip_special_tokens=True).split("assistant")[-1].strip()

# quick look
r0 = val_rows[0] if val_rows else train_rows[0]
print("Q:", r0["question"])
print(solve_image(load_image(r0["image"]), r0["question"], model, tokenizer)[:500])

## 9. Evaluation harness (base vs. tuned)

This is the point of the whole project. The harness parses each model output (preferring the machine-readable `FINAL:` line, with a prose fallback) and scores it against the exact gold labels along separate axes:

- **component_accuracy** — did it count/identify the components right (the perception skill)?
- **Req_accuracy / answer_accuracy** — did it set up and solve correctly (within tolerance vs. the solver)?
- **fabrication_rate** vs. **honest_abstention_rate** — on illegible-value cases, did it invent a number (bad) or correctly say null (good)?

We run it on the **base** model and the **tuned** model over the same held-out set and compare.


In [ ]:
import re, json
def parse_output(text):
    out = {"n_resistors":None,"R_eq":None,"answer_id":None,"answer_current":None,"abstain":None}
    m = re.search(r"FINAL:\s*(\{.*\})", text, re.DOTALL)
    if m:
        try:
            j = json.loads(m.group(1))
            out.update({k: j.get(k, out[k]) for k in out})
            if out["abstain"] is None: out["abstain"] = bool(j.get("abstain", False))
            return out
        except Exception: pass
    low = text.lower()
    out["abstain"] = ("null" in low) or ("not legible" in low) or ("cannot" in low)
    r = re.search(r"r_eq\s*=\s*(\d+(?:\.\d+)?)", low)
    if r: out["R_eq"] = float(r.group(1))
    a = re.search(r"through\s+(\w+)\s*=\s*(\d+(?:\.\d+)?)\s*a", low)
    if a: out["answer_id"], out["answer_current"] = a.group(1).upper(), float(a.group(2))
    n = re.search(r"(\d+)\s*resistor", low)
    if n: out["n_resistors"] = int(n.group(1))
    return out

KNOWN_CONCEPTS = ["ohm's law", "parallel resistor combination", "series resistor combination"]
def parse_concepts(text):
    """Extract the model's declared 'Concepts used: ...' list, normalized to known concepts."""
    import re as _re
    m = _re.search(r"concepts used:\s*(.+)", text.lower())
    if not m: return None
    chunk = m.group(1).split("\n")[0]
    found = set()
    for kc in KNOWN_CONCEPTS:
        key = kc.split()[0] if kc!="ohm's law" else "ohm"
        if key in chunk: found.add(kc)
    return found

def gold_answer_id(gold):
    m = re.search(r"through\s+(\w+)", gold["question"]); return m.group(1) if m else None

def score_record(gold, model_text, rel_tol=0.02):
    p = parse_output(model_text); ga = gold["abstain"]
    res = {"comp_ok":None,"req_ok":None,"answer_ok":None,"abstain_gold":ga,
           "abstain_pred":bool(p["abstain"]),"fabricated":False,"honest_abstain":False,
           "concept_declared":None,"concept_hallucinated":None}
    res["comp_ok"] = (p["n_resistors"] == gold["gold_components"]["resistor"])
    # concept-declaration check (only if the record carries gold concepts)
    gc = gold.get("concepts")
    if gc is not None:
        pc = parse_concepts(model_text)
        gold_set = set(c.lower() for c in gc)
        if pc is None:
            res["concept_declared"] = False; res["concept_hallucinated"] = None
        else:
            res["concept_declared"] = True
            res["concept_hallucinated"] = len(pc - gold_set) > 0   # invoked a concept not needed
    else:
        res["concept_declared"] = None; res["concept_hallucinated"] = None
    if ga:
        gave = (p["answer_current"] is not None) and (not p["abstain"])
        res["fabricated"] = gave
        res["honest_abstain"] = bool(p["abstain"]) and not gave
    else:
        g = gold["gold_values"]["R_eq"]
        res["req_ok"] = (p["R_eq"] is not None and g and abs(p["R_eq"]-g)/g <= rel_tol)
        gi = gold["gold_values"]["branch_currents"].get(gold_answer_id(gold))
        res["answer_ok"] = (p["answer_current"] is not None and gi is not None
                            and abs(p["answer_current"]-gi) <= max(1e-4, abs(gi)*rel_tol))
    return res

def aggregate(results):
    non=[r for r in results if not r["abstain_gold"]]; ab=[r for r in results if r["abstain_gold"]]
    def frac(xs,k):
        xs=[r[k] for r in xs if r[k] is not None]; return round(sum(xs)/len(xs),4) if xs else None
    return {"n_total":len(results),"n_abstain":len(ab),
            "component_accuracy":frac(results,"comp_ok"),
            "Req_accuracy":frac(non,"req_ok"),"answer_accuracy":frac(non,"answer_ok"),
            "fully_correct":round(sum(1 for r in non if r["comp_ok"] and r["req_ok"] and r["answer_ok"])/len(non),4) if non else None,
            "fabrication_rate":frac(ab,"fabricated"),"honest_abstention_rate":frac(ab,"honest_abstain"),
            "concept_declared_rate":frac(results,"concept_declared"),
            "concept_hallucination_rate":frac(results,"concept_hallucinated")}

def evaluate(model, tokenizer, rows, n=None):
    rows = rows[:n] if n else rows
    res=[]
    for r in rows:
        txt = solve_image(load_image(r["image"]), r["question"], model, tokenizer)
        res.append(score_record(r, txt))
    return aggregate(res)
print("eval harness loaded")

In [ ]:
# Base vs tuned on the held-out synthetic val set (use the real-world eval set for the headline number).
EVAL_ROWS = val_rows if val_rows else train_rows[-40:]
EVAL_N = min(60, len(EVAL_ROWS))

# --- tuned (current, fine-tuned model) ---
tuned_metrics = evaluate(model, tokenizer, EVAL_ROWS, EVAL_N)
print("TUNED :", tuned_metrics)

# --- base (reload a fresh, un-tuned model) ---
base_model, base_tok = FastVisionModel.from_pretrained(
    CFG["MODEL"], load_in_4bit=True, use_gradient_checkpointing="unsloth")
FastVisionModel.for_inference(base_model)
base_metrics = evaluate(base_model, base_tok, EVAL_ROWS, EVAL_N)
print("BASE  :", base_metrics)

print("\n=== DELTA (tuned - base) ===")
for k in tuned_metrics:
    if isinstance(tuned_metrics[k],(int,float)) and isinstance(base_metrics.get(k),(int,float)):
        print(f"{k:24s}: {base_metrics[k]:.3f} -> {tuned_metrics[k]:.3f}  ({tuned_metrics[k]-base_metrics[k]:+.3f})")

## 10. Real-world evaluation (the headline number)

Synthetic accuracy overstates real performance. Fill in the hand-labeled real set (`real_eval_TEMPLATE.json` from the data-gen notebook), load it the same way, and run `evaluate(...)` on it. Report **base vs. tuned on the real set** as your main result, and report the synthetic→real gap honestly.

In [ ]:
real_path = os.path.join(CFG["DATA_DIR"], "real_eval.json")   # your filled-in file
if os.path.exists(real_path):
    real_rows = json.load(open(real_path))
    for r in real_rows: r.setdefault("abstain", False)
    print("REAL tuned:", evaluate(model, tokenizer, real_rows))
    print("REAL base :", evaluate(base_model, base_tok, real_rows))
else:
    print("No real_eval.json yet - hand-label a few dozen real diagrams to get the transfer number.")

## 11. Save / push the adapter

In [ ]:
model.save_pretrained(CFG["OUT_DIR"]); tokenizer.save_pretrained(CFG["OUT_DIR"])
print("saved LoRA adapter to", CFG["OUT_DIR"])
# To the Hub:
# from huggingface_hub import login; login()
# model.push_to_hub("your-username/circuitsight-qwen2.5vl-3b")
# tokenizer.push_to_hub("your-username/circuitsight-qwen2.5vl-3b")
# Merged 16-bit for deployment:
# model.save_pretrained_merged("circuitsight_merged", tokenizer)

## 12. Notes & troubleshooting

- **`could not make a flat list of images`** → ensure `to_conversation` passes PIL images (it does) and exactly one image per sample.
- **`max_length` vs `max_seq_length`** → newer TRL uses `max_length` (used here). If your TRL errors, rename it to `max_seq_length`.
- **OOM mid-run** → lower `MAX_IMAGE_PX` (768→512), keep `use_gradient_checkpointing="unsloth"`, `BATCH=1`, raise `GRAD_ACCUM`; last resort, `finetune_vision_layers=False` or a 2B model.
- **Tuned barely beats base** → concentrate the eval on harder cases (more components, parallel blocks) and on the abstention subset, where the base fails most; check the loss actually dropped; add render variety in the data.
- **Scaling** → build data for 50k but train up from ~10-15k, watching the section-9 curve; stop when it flattens.


## 13. (v2) DPO preference tuning — build this AFTER v1 succeeds

> **Do not build/run this yet.** DPO is a *second stage that runs on top of the v1 fine-tuned model*, not an alternative to it. It only becomes meaningful — and measurable — once v1 has posted a solid base-vs-tuned gain. Running it on a weak v1 model just adds noise you can't interpret.

**When to come back here:** after section 9 shows the tuned model clearly beating the base on component / R_eq / answer accuracy (and ideally on the real-world set in section 10).

**What it will do when built:**
- Load `train_pairs.jsonl` (already produced by the data notebook when `INCLUDE_MISTAKES=True`).
- Form preference pairs: `correct_solution` = *chosen*, `wrong_solution` = *rejected*, same image + question as the prompt.
- Run DPO on top of the v1 LoRA adapter (Unsloth supports vision DPO), which sharpens the model *away* from the exact mistakes in the pairs (parallel-as-series, omitted branch, Ohm's-law flip, misread value).
- Re-run the section-9 harness to check DPO improved spec adherence *beyond* SFT alone — especially the fabrication / setup-error cases.

**Why it's deferred, not written now:** vision DPO has its own trainer, a reference-model copy (tighter VRAM than SFT), and settings that depend on what v1 actually produced and which model/hardware the Day-1 smoke test landed on. Writing it before v1 exists risks writing it twice. The data is already waiting, so nothing is blocked by deferring.